# FREIGHT-MNet: M0.5 vs MLP Error Analysis Notebook

This notebook performs a row-level diagnostic comparison between the **M0.5 hybrid LightGBM baselines** and the **MLP residual baselines** for the East-Plus-Gulf temporal test setting.

The analysis is designed for the current FREIGHT-MNet experiment protocol:

- **Task:** predict annual truck travel-time quantiles for FAF OD-year edges.
- **Labels:** `truck_q25`, `truck_q50`, and `truck_q75`.
- **Main split:** train on 2018--2022, validate on 2023, test on 2024.
- **Main scope:** East-Plus-Gulf.
- **Core comparison:** history-augmented LightGBM residual models versus history-augmented MLP residual models.

The notebook intentionally focuses on **diagnostics**, not model training. It loads saved prediction artifacts, normalizes their schemas, computes comparable metrics, and produces segment-level error tables, paired model-comparison tables, and optional diagnostic plots.

## 1. Environment setup and imports

This notebook avoids `scikit-learn` and does not require `matplotlib` to complete the tabular analysis. Matplotlib is imported only as an optional plotting dependency. If plotting is unavailable, the notebook still saves all CSV/Parquet diagnostic outputs.

The notebook also avoids subprocess preflight checks because earlier runs showed that subprocess kernels could pick up a different NumPy installation from the active notebook kernel.

In [2]:
from __future__ import annotations

import json
import math
import os
import platform
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

# Disable optional pandas acceleration modules that can trigger binary-compatibility
# issues in mixed NumPy environments. This does not affect correctness.
pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)

# Matplotlib is optional. All core outputs are CSV/Parquet tables.
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except Exception as exc:  # pragma: no cover - plotting is optional.
    HAS_MATPLOTLIB = False
    plt = None
    print(f"Matplotlib is unavailable. Plot generation will be skipped. Reason: {exc}")

print("Python executable:", os.sys.executable)
print("Python version   :", platform.python_version())
print("Platform         :", platform.platform())
print("NumPy            :", np.__version__)
print("Pandas           :", pd.__version__)
print("Matplotlib       :", "available" if HAS_MATPLOTLIB else "not available")

Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Python version   : 3.11.5
Platform         : Windows-10-10.0.26200-SP0
NumPy            : 2.4.5
Pandas           : 2.3.3
Matplotlib       : available


## 2. Experiment configuration

Only this cell usually needs manual editing. The default paths match the project layout used by the previous M0.5 and MLP notebooks.

The notebook searches for prediction files inside these directories:

- `Data/10_experiments/m05_hybrid_baselines_v1_notebook/east_plus_gulf/`
- `Data/10_experiments/mlp_residual_v1_notebook/east_plus_gulf/`

The strict model-name checks are important. They prevent accidentally comparing the MLP predictions against an old M0 baseline prediction file with the same filename.

In [3]:
@dataclass(frozen=True)
class ExperimentConfig:
    """Configuration for the M0.5-vs-MLP diagnostic analysis."""

    # Root directory of the FREIGHT-MNet data tree.
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")

    # Main experiment scope.
    scope: str = "east_plus_gulf"

    # Upstream experiment run names.
    m05_run_name: str = "m05_hybrid_baselines_v1_notebook"
    mlp_run_name: str = "mlp_residual_v1_notebook"

    # Output run name for this analysis notebook.
    analysis_run_name: str = "m05_vs_mlp_error_analysis_v1_notebook"

    # Required temporal splits for the main experiment.
    validation_year: int = 2023
    test_year: int = 2024

    # Whether the notebook should stop if the M0.5 prediction file is missing
    # or does not contain the expected M0.5 models. Keep this True for final experiments.
    require_valid_m05_predictions: bool = True

    # Whether the notebook should stop if the MLP prediction file is missing
    # or does not contain the expected MLP models. Keep this True for final experiments.
    require_valid_mlp_predictions: bool = True

    # Recompute a common weight from obs_weight_sum for all models so that M0.5 and
    # MLP weighted metrics are exactly comparable in this diagnostic notebook.
    recompute_common_sample_weight: bool = True
    weight_clip_min: float = 0.05
    weight_clip_max: float = 20.0

    # Number of top row-level disagreement cases to save for manual inspection.
    top_case_count: int = 100

    # Optional plot generation.
    make_plots: bool = True


CFG = ExperimentConfig()
CFG

ExperimentConfig(data_root=WindowsPath('E:/NetworkOptimization/pythonProject1/Data'), scope='east_plus_gulf', m05_run_name='m05_hybrid_baselines_v1_notebook', mlp_run_name='mlp_residual_v1_notebook', analysis_run_name='m05_vs_mlp_error_analysis_v1_notebook', validation_year=2023, test_year=2024, require_valid_m05_predictions=True, require_valid_mlp_predictions=True, recompute_common_sample_weight=True, weight_clip_min=0.05, weight_clip_max=20.0, top_case_count=100, make_plots=True)

## 3. Path management

This cell constructs all input and output paths from the configuration. The notebook writes results to a dedicated analysis directory and keeps outputs separate from training artifacts.

In [4]:
@dataclass(frozen=True)
class ExperimentPaths:
    """Resolved input and output paths for the analysis notebook."""

    data_root: Path
    supervised_path: Path
    manifest_path: Path
    m05_output_dir: Path
    mlp_output_dir: Path
    analysis_output_dir: Path
    table_dir: Path
    plot_dir: Path
    report_dir: Path


def build_paths(cfg: ExperimentConfig) -> ExperimentPaths:
    """Build canonical project paths from the experiment configuration."""
    model_ready_dir = cfg.data_root / "08_processed" / "model_ready"
    supervised_path = model_ready_dir / f"freight_mnet_supervised_edges_2018_2024_{cfg.scope}.parquet"
    manifest_path = model_ready_dir / "_metadata" / "freight_mnet_supervised_feature_manifest.json"

    m05_output_dir = cfg.data_root / "10_experiments" / cfg.m05_run_name / cfg.scope
    mlp_output_dir = cfg.data_root / "10_experiments" / cfg.mlp_run_name / cfg.scope
    analysis_output_dir = cfg.data_root / "10_experiments" / cfg.analysis_run_name / cfg.scope

    return ExperimentPaths(
        data_root=cfg.data_root,
        supervised_path=supervised_path,
        manifest_path=manifest_path,
        m05_output_dir=m05_output_dir,
        mlp_output_dir=mlp_output_dir,
        analysis_output_dir=analysis_output_dir,
        table_dir=analysis_output_dir / "tables",
        plot_dir=analysis_output_dir / "plots",
        report_dir=analysis_output_dir / "reports",
    )


def ensure_output_dirs(paths: ExperimentPaths) -> None:
    """Create output directories used by this notebook."""
    for path in [paths.analysis_output_dir, paths.table_dir, paths.plot_dir, paths.report_dir]:
        path.mkdir(parents=True, exist_ok=True)


PATHS = build_paths(CFG)
ensure_output_dirs(PATHS)

print("Supervised table path:", PATHS.supervised_path)
print("Feature manifest path:", PATHS.manifest_path)
print("M0.5 output directory:", PATHS.m05_output_dir)
print("MLP output directory:", PATHS.mlp_output_dir)
print("Analysis output directory:", PATHS.analysis_output_dir)

Supervised table path: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
Feature manifest path: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json
M0.5 output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf
MLP output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf
Analysis output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_vs_mlp_error_analysis_v1_notebook\east_plus_gulf


## 4. Constants, model sets, and schema rules

The analysis compares model families using standardized names. The code supports extra models, but it checks that the core M0.5 and MLP models are present before producing paired comparisons.

Primary comparison pairs:

1. `residual_lgbm_prior_features` vs `mlp_residual_current_features`
2. `prior_feature_lgbm_direct` vs `mlp_residual_prior_features`
3. `od_historical_prior` vs each residual MLP

These pairings are chosen because M0.5 established history-augmented LightGBM as the strongest tabular baseline, while the MLP experiment showed that residual neural learning is the strongest neural no-graph baseline.

In [5]:
LABEL_COLUMNS = ["q25", "q50", "q75"]
TAUS = {"q25": 0.25, "q50": 0.50, "q75": 0.75}
KEY_COLUMNS = ["split", "faf_orig", "faf_dest", "year"]

EXPECTED_M05_MODELS = {
    "od_historical_prior",
    "lightgbm_direct",
    "hist_lgbm_ensemble_global",
    "hist_lgbm_ensemble_per_quantile",
    "residual_lgbm_current_features",
    "residual_lgbm_prior_features",
    "prior_feature_lgbm_direct",
}

EXPECTED_MLP_MODELS = {
    "global_train_median",
    "od_historical_prior",
    "mlp_raw_current_features",
    "mlp_prior_direct",
    "mlp_residual_current_features",
    "mlp_residual_prior_features",
}

# Models used for default leaderboards and segment plots.
DEFAULT_FOCUS_MODELS = [
    "M05::residual_lgbm_prior_features",
    "M05::prior_feature_lgbm_direct",
    "MLP::mlp_residual_current_features",
    "MLP::mlp_residual_prior_features",
    "MLP::od_historical_prior",
]

# Pair definitions use the convention: positive deltas mean the MLP model is better.
PAIR_DEFINITIONS = [
    {
        "pair_name": "M05_residual_prior_vs_MLP_residual_current",
        "m05_model": "residual_lgbm_prior_features",
        "mlp_model": "mlp_residual_current_features",
    },
    {
        "pair_name": "M05_prior_feature_direct_vs_MLP_residual_prior",
        "m05_model": "prior_feature_lgbm_direct",
        "mlp_model": "mlp_residual_prior_features",
    },
    {
        "pair_name": "Historical_prior_vs_MLP_residual_current",
        "m05_model": "od_historical_prior",
        "mlp_model": "mlp_residual_current_features",
    },
    {
        "pair_name": "Historical_prior_vs_MLP_residual_prior",
        "m05_model": "od_historical_prior",
        "mlp_model": "mlp_residual_prior_features",
    },
]

## 5. File resolution and artifact validation

This section locates prediction files and validates that they contain the expected models. This guard is necessary because several earlier notebooks save files with generic names such as `predictions_val_test.parquet`.

If a candidate file contains only old M0 models such as `ridge`, `elasticnet`, and `lightgbm_quantile`, the notebook rejects it as an M0.5 artifact.

In [6]:
def resolve_first_existing_path(candidates: Sequence[Path], label: str) -> Optional[Path]:
    """Return the first existing path from a candidate list."""
    print(f"\nSearching for {label} candidates:")
    for path in candidates:
        status = "FOUND" if path.exists() else "missing"
        print(f"  [{status}] {path}")
        if path.exists():
            return path
    return None


def read_model_names_from_parquet(path: Path) -> List[str]:
    """Read unique model names from a prediction parquet file."""
    df_head = pd.read_parquet(path, columns=["model"])
    return sorted(map(str, df_head["model"].dropna().unique().tolist()))


def validate_prediction_file(
    path: Optional[Path],
    expected_models: set[str],
    label: str,
    require_valid: bool,
) -> Optional[Path]:
    """Validate that a prediction file exists and contains expected model names."""
    if path is None:
        message = f"No candidate prediction file was found for {label}."
        if require_valid:
            raise FileNotFoundError(message)
        warnings.warn(message)
        return None

    actual_models = set(read_model_names_from_parquet(path))
    missing = expected_models - actual_models
    overlap = expected_models & actual_models

    print(f"\n{label} prediction file:", path)
    print(f"Detected models ({len(actual_models)}):", sorted(actual_models))
    print(f"Expected-model overlap ({len(overlap)}):", sorted(overlap))

    if missing:
        message = (
            f"{label} prediction file does not contain all expected models. "
            f"Missing: {sorted(missing)}. File: {path}"
        )
        if require_valid:
            raise ValueError(message)
        warnings.warn(message)

    return path


m05_prediction_candidates = [
    PATHS.m05_output_dir / "predictions_val_test_m05_hybrid.parquet",
    PATHS.m05_output_dir / "predictions_val_test_m05.parquet",
    PATHS.m05_output_dir / "predictions_val_test_hybrid.parquet",
    PATHS.m05_output_dir / "predictions_val_test.parquet",
]

mlp_prediction_candidates = [
    PATHS.mlp_output_dir / "predictions_val_test_mlp_residual.parquet",
    PATHS.mlp_output_dir / "predictions_val_test.parquet",
]

m05_prediction_path = validate_prediction_file(
    resolve_first_existing_path(m05_prediction_candidates, "M0.5 prediction file"),
    EXPECTED_M05_MODELS,
    "M0.5",
    CFG.require_valid_m05_predictions,
)

mlp_prediction_path = validate_prediction_file(
    resolve_first_existing_path(mlp_prediction_candidates, "MLP prediction file"),
    EXPECTED_MLP_MODELS,
    "MLP",
    CFG.require_valid_mlp_predictions,
)


Searching for M0.5 prediction file candidates:
  [missing] E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\predictions_val_test_m05_hybrid.parquet
  [missing] E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\predictions_val_test_m05.parquet
  [missing] E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\predictions_val_test_hybrid.parquet
  [FOUND] E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\predictions_val_test.parquet

M0.5 prediction file: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\predictions_val_test.parquet
Detected models (8): ['global_train_median', 'hist_lgbm_ensemble_global', 'hist_lgbm_ensemble_per_quantile', 'lightgbm_direct', 'od_historical_prior', 'prior_feature_lgbm_direct', 'resid

## 6. Load supervised data and feature manifest

The supervised model-ready table provides the row keys, true labels, observation-support diagnostics, and segmentation features used for error analysis. The feature manifest is loaded only for validation and documentation.

The predictive feature vector should not include FMI aggregation diagnostics such as `n_fmi_county_pairs` or `obs_weight_sum`. However, this diagnostic notebook can use those fields for segment analysis and sample weighting.

In [7]:
def read_json(path: Path) -> dict:
    """Read a UTF-8 JSON file."""
    with path.open("r", encoding="utf-8") as fp:
        return json.load(fp)


def normalize_faf_code(series: pd.Series) -> pd.Series:
    """Normalize FAF zone codes as zero-padded three-character strings."""
    return series.astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(3)


def load_supervised_table(paths: ExperimentPaths, cfg: ExperimentConfig) -> pd.DataFrame:
    """Load and validate the supervised model-ready table."""
    if not paths.supervised_path.exists():
        raise FileNotFoundError(f"Supervised table not found: {paths.supervised_path}")

    df = pd.read_parquet(paths.supervised_path)
    required = {"faf_orig", "faf_dest", "year", "truck_q25", "truck_q50", "truck_q75"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Supervised table is missing required columns: {sorted(missing)}")

    df = df.copy()
    df["faf_orig"] = normalize_faf_code(df["faf_orig"])
    df["faf_dest"] = normalize_faf_code(df["faf_dest"])
    df["year"] = df["year"].astype(int)
    df["split"] = np.select(
        [df["year"].between(2018, 2022), df["year"].eq(cfg.validation_year), df["year"].eq(cfg.test_year)],
        ["train", "val", "test"],
        default="other",
    )
    df["faf_od"] = df["faf_orig"] + "->" + df["faf_dest"]
    df["true_iqr"] = df["truck_q75"].astype(float) - df["truck_q25"].astype(float)

    return df


manifest = read_json(PATHS.manifest_path) if PATHS.manifest_path.exists() else {}
supervised_df = load_supervised_table(PATHS, CFG)

print("Supervised shape:", supervised_df.shape)
print("Year counts:")
print(supervised_df["year"].value_counts().sort_index())
print("Split counts:")
print(supervised_df["split"].value_counts().sort_index())
print("Manifest keys:", sorted(manifest.keys()) if isinstance(manifest, dict) else type(manifest))

Supervised shape: (73972, 93)
Year counts:
year
2018     9936
2019    10704
2020    10761
2021    10721
2022    10651
2023    10625
2024    10574
Name: count, dtype: int64
Split counts:
split
test     10574
train    52773
val      10625
Name: count, dtype: int64
Manifest keys: ['excluded_fmi_aux_columns', 'feature_columns', 'id_columns', 'label_columns', 'loss_weight_columns', 'note']


## 7. Normalize prediction schemas

M0.5 and MLP prediction files use different column names. This cell converts both into a common long schema:

- `source`: `M05` or `MLP`
- `model_id`: source-prefixed model identifier
- true labels: `true_q25`, `true_q50`, `true_q75`
- predictions: `pred_q25`, `pred_q50`, `pred_q75`
- row diagnostics: absolute errors, bias errors, pinball losses, IQR error, and common sample weight

This makes all downstream metrics and paired comparisons source-agnostic.

In [8]:
def coalesce_column(df: pd.DataFrame, candidates: Sequence[str], default: object = np.nan) -> pd.Series:
    """Return the first existing candidate column as a Series."""
    for col in candidates:
        if col in df.columns:
            return df[col]
    return pd.Series(default, index=df.index)


def normalize_prediction_table(raw: pd.DataFrame, source: str) -> pd.DataFrame:
    """Normalize M0.5 or MLP predictions to a common schema."""
    df = pd.DataFrame(index=raw.index)
    df["source"] = source
    df["model"] = raw["model"].astype(str)
    df["variant"] = coalesce_column(raw, ["variant"], "postprocessed").fillna("postprocessed").astype(str)
    df["split"] = coalesce_column(raw, ["split", "eval_split"]).astype(str)
    df["faf_orig"] = normalize_faf_code(coalesce_column(raw, ["faf_orig", "origin", "orig_faf"]))
    df["faf_dest"] = normalize_faf_code(coalesce_column(raw, ["faf_dest", "destination", "dest_faf"]))
    df["year"] = coalesce_column(raw, ["year"]).astype(int)
    df["faf_od"] = df["faf_orig"] + "->" + df["faf_dest"]

    # True labels.
    df["true_q25"] = pd.to_numeric(coalesce_column(raw, ["true_q25", "actual_truck_q25", "truck_q25"]), errors="coerce")
    df["true_q50"] = pd.to_numeric(coalesce_column(raw, ["true_q50", "actual_truck_q50", "truck_q50"]), errors="coerce")
    df["true_q75"] = pd.to_numeric(coalesce_column(raw, ["true_q75", "actual_truck_q75", "truck_q75"]), errors="coerce")

    # Postprocessed predictions.
    df["pred_q25"] = pd.to_numeric(coalesce_column(raw, ["pred_q25", "pred_truck_q25"]), errors="coerce")
    df["pred_q50"] = pd.to_numeric(coalesce_column(raw, ["pred_q50", "pred_truck_q50"]), errors="coerce")
    df["pred_q75"] = pd.to_numeric(coalesce_column(raw, ["pred_q75", "pred_truck_q75"]), errors="coerce")

    # Raw predictions are optional. They are used to diagnose crossing before postprocessing.
    df["pred_raw_q25"] = pd.to_numeric(coalesce_column(raw, ["pred_raw_q25", "pred_raw_truck_q25", "pred_q25", "pred_truck_q25"]), errors="coerce")
    df["pred_raw_q50"] = pd.to_numeric(coalesce_column(raw, ["pred_raw_q50", "pred_raw_truck_q50", "pred_q50", "pred_truck_q50"]), errors="coerce")
    df["pred_raw_q75"] = pd.to_numeric(coalesce_column(raw, ["pred_raw_q75", "pred_raw_truck_q75", "pred_q75", "pred_truck_q75"]), errors="coerce")

    # Observation-support diagnostics are not predictive features here; they are diagnostic variables.
    df["n_fmi_county_pairs"] = pd.to_numeric(coalesce_column(raw, ["n_fmi_county_pairs"], np.nan), errors="coerce")
    df["obs_weight_sum"] = pd.to_numeric(coalesce_column(raw, ["obs_weight_sum"], np.nan), errors="coerce")

    # MLP predictions include historical prior columns. M0.5 predictions may not.
    for q in ["q25", "q50", "q75"]:
        df[f"hist_prior_{q}"] = pd.to_numeric(coalesce_column(raw, [f"hist_prior_{q}", f"hist_truck_{q}"]), errors="coerce")

    df["model_id"] = df["source"] + "::" + df["model"]
    return df


def add_error_columns(df: pd.DataFrame, cfg: ExperimentConfig) -> pd.DataFrame:
    """Add row-level errors, pinball losses, interval diagnostics, and common weights."""
    out = df.copy()

    for q in LABEL_COLUMNS:
        out[f"error_{q}"] = out[f"pred_{q}"] - out[f"true_{q}"]
        out[f"abs_error_{q}"] = out[f"error_{q}"].abs()
        tau = TAUS[q]
        residual = out[f"true_{q}"] - out[f"pred_{q}"]
        out[f"pinball_{q}"] = np.maximum(tau * residual, (tau - 1.0) * residual)

    out["pinball_mean_row"] = out[[f"pinball_{q}" for q in LABEL_COLUMNS]].mean(axis=1)
    out["true_iqr"] = out["true_q75"] - out["true_q25"]
    out["pred_iqr"] = out["pred_q75"] - out["pred_q25"]
    out["abs_error_iqr"] = (out["pred_iqr"] - out["true_iqr"]).abs()
    out["raw_crossing"] = (out["pred_raw_q25"] > out["pred_raw_q50"]) | (out["pred_raw_q50"] > out["pred_raw_q75"])
    out["post_crossing"] = (out["pred_q25"] > out["pred_q50"]) | (out["pred_q50"] > out["pred_q75"])
    out["q50_inside_pred_interval"] = (out["true_q50"] >= out["pred_q25"]) & (out["true_q50"] <= out["pred_q75"])

    if cfg.recompute_common_sample_weight:
        obs = out["obs_weight_sum"].fillna(0).clip(lower=0).astype(float)
        w = np.log1p(obs)
        positive_mean = w[w > 0].mean()
        if not np.isfinite(positive_mean) or positive_mean <= 0:
            w = pd.Series(1.0, index=out.index)
        else:
            w = w / positive_mean
        out["sample_weight_common"] = w.clip(cfg.weight_clip_min, cfg.weight_clip_max).astype(float)
    else:
        out["sample_weight_common"] = pd.to_numeric(coalesce_column(out, ["sample_weight"], 1.0), errors="coerce").fillna(1.0)

    return out


m05_raw = pd.read_parquet(m05_prediction_path) if m05_prediction_path is not None else pd.DataFrame()
mlp_raw = pd.read_parquet(mlp_prediction_path) if mlp_prediction_path is not None else pd.DataFrame()

m05_pred = add_error_columns(normalize_prediction_table(m05_raw, "M05"), CFG) if not m05_raw.empty else pd.DataFrame()
mlp_pred = add_error_columns(normalize_prediction_table(mlp_raw, "MLP"), CFG) if not mlp_raw.empty else pd.DataFrame()

combined_pred = pd.concat([m05_pred, mlp_pred], ignore_index=True)
combined_pred = combined_pred[combined_pred["split"].isin(["val", "test"])].copy()

print("M0.5 normalized shape:", m05_pred.shape)
print("MLP normalized shape:", mlp_pred.shape)
print("Combined shape:", combined_pred.shape)
print("Combined model IDs:")
print(pd.Series(sorted(combined_pred["model_id"].unique())).to_string(index=False))

M0.5 normalized shape: (169592, 40)
MLP normalized shape: (127194, 40)
Combined shape: (296786, 40)
Combined model IDs:
            M05::global_train_median
      M05::hist_lgbm_ensemble_global
M05::hist_lgbm_ensemble_per_quantile
                M05::lightgbm_direct
            M05::od_historical_prior
      M05::prior_feature_lgbm_direct
 M05::residual_lgbm_current_features
   M05::residual_lgbm_prior_features
            MLP::global_train_median
               MLP::mlp_prior_direct
       MLP::mlp_raw_current_features
  MLP::mlp_residual_current_features
    MLP::mlp_residual_prior_features
            MLP::od_historical_prior


C:\Users\Nick_James\AppData\Local\Temp\ipykernel_24836\2882469292.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_pred = pd.concat([m05_pred, mlp_pred], ignore_index=True)


## 8. Add segmentation features from the supervised table

This cell merges row-level predictions with useful diagnostic features from the model-ready supervised table.

Examples include:

- `same_faf_zone`
- `has_rail_demand`
- `has_multimodal_demand`
- `n_modes_with_positive_tons`
- total demand and modal share features
- observation-support fields such as `n_fmi_county_pairs` and `obs_weight_sum`

These fields support stress, sparse, demand-mode, and corridor-type diagnostics.

In [9]:
SEGMENT_FEATURE_CANDIDATES = [
    "same_faf_zone",
    "has_truck_demand",
    "has_rail_demand",
    "has_multimodal_demand",
    "n_modes_with_positive_tons",
    "log1p_total_tons_modes_1_2_5",
    "log1p_total_tmiles_modes_1_2_5",
    "log1p_total_value_modes_1_2_5",
    "tons_share_truck",
    "tons_share_rail",
    "tons_share_multimodal",
    "tmiles_share_truck",
    "tmiles_share_rail",
    "tmiles_share_multimodal",
    "rail_or_multimodal_tons_share",
    "truck_to_rail_tons_ratio",
    "truck_to_multimodal_tons_ratio",
    "orig_n_counties",
    "dest_n_counties",
    "od_n_counties_sum",
    "od_n_counties_abs_diff",
    "orig_east_plus_gulf_county_share",
    "dest_east_plus_gulf_county_share",
    "od_east_plus_gulf_share_mean",
    "n_fmi_county_pairs",
    "obs_weight_sum",
]


def build_segmentation_frame(supervised: pd.DataFrame) -> pd.DataFrame:
    """Build a unique key-level feature table for segmentation."""
    available_features = [c for c in SEGMENT_FEATURE_CANDIDATES if c in supervised.columns]
    keep_cols = ["split", "faf_orig", "faf_dest", "year", "truck_q25", "truck_q50", "truck_q75", "true_iqr"] + available_features
    seg = supervised.loc[supervised["split"].isin(["val", "test"]), keep_cols].copy()
    seg = seg.drop_duplicates(KEY_COLUMNS)

    # Rename true labels to avoid collisions with normalized prediction columns.
    seg = seg.rename(columns={"truck_q25": "seg_true_q25", "truck_q50": "seg_true_q50", "truck_q75": "seg_true_q75", "true_iqr": "seg_true_iqr"})
    return seg


def merge_segmentation_features(pred: pd.DataFrame, supervised: pd.DataFrame) -> pd.DataFrame:
    """Merge segmentation features into the prediction table."""
    seg = build_segmentation_frame(supervised)
    merged = pred.merge(seg, on=KEY_COLUMNS, how="left", validate="many_to_one")

    # Prefer prediction-file diagnostics if present; otherwise use supervised-table diagnostics.
    for col in ["n_fmi_county_pairs", "obs_weight_sum"]:
        col_x, col_y = f"{col}_x", f"{col}_y"
        if col_x in merged.columns and col_y in merged.columns:
            merged[col] = merged[col_x].combine_first(merged[col_y])
            merged = merged.drop(columns=[col_x, col_y])

    return merged


combined_pred = merge_segmentation_features(combined_pred, supervised_df)

print("Shape after segmentation merge:", combined_pred.shape)
print("Available segmentation columns:")
print([c for c in SEGMENT_FEATURE_CANDIDATES if c in combined_pred.columns])

Shape after segmentation merge: (296786, 68)
Available segmentation columns:
['same_faf_zone', 'has_truck_demand', 'has_rail_demand', 'has_multimodal_demand', 'n_modes_with_positive_tons', 'log1p_total_tons_modes_1_2_5', 'log1p_total_tmiles_modes_1_2_5', 'log1p_total_value_modes_1_2_5', 'tons_share_truck', 'tons_share_rail', 'tons_share_multimodal', 'tmiles_share_truck', 'tmiles_share_rail', 'tmiles_share_multimodal', 'rail_or_multimodal_tons_share', 'truck_to_rail_tons_ratio', 'truck_to_multimodal_tons_ratio', 'orig_n_counties', 'dest_n_counties', 'od_n_counties_sum', 'od_n_counties_abs_diff', 'orig_east_plus_gulf_county_share', 'dest_east_plus_gulf_county_share', 'od_east_plus_gulf_share_mean', 'n_fmi_county_pairs', 'obs_weight_sum']


## 9. Metric utilities

These functions compute model-level and segment-level diagnostics. The metrics align with the FREIGHT-MNet evaluation protocol:

- quantile MAE for q25/q50/q75
- pinball loss at q25/q50/q75 and averaged across quantiles
- IQR MAE
- top-decile q75 stress error
- sparse-label q75 error
- interval validity and crossing diagnostics

In [10]:
def weighted_mean(values: pd.Series, weights: Optional[pd.Series] = None) -> float:
    """Compute a robust weighted mean."""
    x = pd.to_numeric(values, errors="coerce").astype(float)
    mask = np.isfinite(x)
    if weights is None:
        return float(x[mask].mean()) if mask.any() else np.nan

    w = pd.to_numeric(weights, errors="coerce").astype(float)
    mask = mask & np.isfinite(w) & (w > 0)
    if not mask.any():
        return np.nan
    return float(np.average(x[mask], weights=w[mask]))


def summarize_prediction_group(group: pd.DataFrame) -> Dict[str, float]:
    """Summarize prediction errors for one model/split or one model/segment group."""
    w = group["sample_weight_common"] if "sample_weight_common" in group.columns else None
    out: Dict[str, float] = {
        "n_rows": int(len(group)),
        "mae_q25": weighted_mean(group["abs_error_q25"]),
        "mae_q50": weighted_mean(group["abs_error_q50"]),
        "mae_q75": weighted_mean(group["abs_error_q75"]),
        "rmse_q50": float(np.sqrt(np.nanmean(np.square(group["error_q50"].astype(float))))),
        "pinball_q25": weighted_mean(group["pinball_q25"]),
        "pinball_q50": weighted_mean(group["pinball_q50"]),
        "pinball_q75": weighted_mean(group["pinball_q75"]),
        "pinball_mean": weighted_mean(group["pinball_mean_row"]),
        "weighted_pinball_mean": weighted_mean(group["pinball_mean_row"], w),
        "iqr_mae": weighted_mean(group["abs_error_iqr"]),
        "weighted_iqr_mae": weighted_mean(group["abs_error_iqr"], w),
        "bias_q25": weighted_mean(group["error_q25"]),
        "bias_q50": weighted_mean(group["error_q50"]),
        "bias_q75": weighted_mean(group["error_q75"]),
        "raw_crossing_rate": weighted_mean(group["raw_crossing"].astype(float)),
        "post_crossing_rate": weighted_mean(group["post_crossing"].astype(float)),
        "q50_inside_pred_interval_rate": weighted_mean(group["q50_inside_pred_interval"].astype(float)),
    }
    return out


def summarize_by(df: pd.DataFrame, group_cols: Sequence[str]) -> pd.DataFrame:
    """Summarize prediction metrics by one or more grouping columns."""
    records = []
    for keys, group in df.groupby(list(group_cols), dropna=False, sort=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        row.update(summarize_prediction_group(group))
        records.append(row)
    return pd.DataFrame(records)


metrics_by_model_split = summarize_by(combined_pred, ["source", "model", "model_id", "variant", "split"])
leaderboard_test = metrics_by_model_split.loc[metrics_by_model_split["split"].eq("test")].copy()
leaderboard_test = leaderboard_test.sort_values("pinball_mean", ascending=True).reset_index(drop=True)
leaderboard_test.insert(0, "rank_by_test_pinball", np.arange(1, len(leaderboard_test) + 1))

print("Combined test leaderboard:")
print(leaderboard_test[["rank_by_test_pinball", "model_id", "pinball_mean", "weighted_pinball_mean", "mae_q50", "mae_q75", "iqr_mae"]].to_string(index=False))

Combined test leaderboard:
 rank_by_test_pinball                             model_id  pinball_mean  weighted_pinball_mean    mae_q50     mae_q75    iqr_mae
                    1    M05::residual_lgbm_prior_features    131.187824             116.200505 230.953405  451.623436 384.731525
                    2       M05::prior_feature_lgbm_direct    131.304712             116.381888 231.615675  449.055148 381.889820
                    3  M05::residual_lgbm_current_features    132.558496             117.679036 230.401410  465.240711 395.618681
                    4   MLP::mlp_residual_current_features    135.933606             120.340749 242.287672  455.564743 376.481643
                    5     MLP::mlp_residual_prior_features    138.905682             123.590372 239.137838  458.999341 375.083455
                    6       M05::hist_lgbm_ensemble_global    146.728685             130.655899 231.257504  462.529352 370.897768
                    7 M05::hist_lgbm_ensemble_per_quantile    1

## 10. Add global stress and sparse indicators

The stress split is defined on the 2024 test set using the top 10% of the true `q75` label. The sparse-label split uses low observation support based on `n_fmi_county_pairs`.

These indicators are computed at the OD-year row level, then merged back into all model predictions.

In [11]:
def add_global_diagnostic_indicators(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """Add stress/sparse indicators based on unique OD-year rows."""
    out = df.copy()
    unique_test = out.loc[out["split"].eq("test"), KEY_COLUMNS + ["true_q75", "true_iqr", "n_fmi_county_pairs", "obs_weight_sum"]].drop_duplicates(KEY_COLUMNS)

    thresholds: Dict[str, float] = {}
    thresholds["test_true_q75_top10_threshold"] = float(unique_test["true_q75"].quantile(0.90))
    thresholds["test_true_iqr_top10_threshold"] = float(unique_test["true_iqr"].quantile(0.90))
    thresholds["test_n_fmi_bottom25_threshold"] = float(unique_test["n_fmi_county_pairs"].quantile(0.25))
    thresholds["test_obs_weight_bottom25_threshold"] = float(unique_test["obs_weight_sum"].quantile(0.25))

    key_flags = unique_test[KEY_COLUMNS].copy()
    key_flags["is_test_q75_top10"] = unique_test["true_q75"] >= thresholds["test_true_q75_top10_threshold"]
    key_flags["is_test_iqr_top10"] = unique_test["true_iqr"] >= thresholds["test_true_iqr_top10_threshold"]
    key_flags["is_test_sparse_n_fmi_bottom25"] = unique_test["n_fmi_county_pairs"] <= thresholds["test_n_fmi_bottom25_threshold"]
    key_flags["is_test_low_weight_bottom25"] = unique_test["obs_weight_sum"] <= thresholds["test_obs_weight_bottom25_threshold"]

    out = out.merge(key_flags, on=KEY_COLUMNS, how="left", validate="many_to_one")
    for col in ["is_test_q75_top10", "is_test_iqr_top10", "is_test_sparse_n_fmi_bottom25", "is_test_low_weight_bottom25"]:
        out[col] = out[col].fillna(False).astype(bool)

    return out, thresholds


combined_pred, diagnostic_thresholds = add_global_diagnostic_indicators(combined_pred)
print(json.dumps(diagnostic_thresholds, indent=2))

stress_summary = summarize_by(
    combined_pred.loc[combined_pred["split"].eq("test") & combined_pred["is_test_q75_top10"]],
    ["source", "model", "model_id", "variant", "split"],
)
stress_summary = stress_summary.sort_values("mae_q75").reset_index(drop=True)
print("\nTop-q75 stress summary:")
print(stress_summary[["model_id", "n_rows", "mae_q75", "pinball_mean", "iqr_mae"]].head(10).to_string(index=False))

{
  "test_true_q75_top10_threshold": 6194.4000000000015,
  "test_true_iqr_top10_threshold": 3649.761,
  "test_n_fmi_bottom25_threshold": 30.0,
  "test_obs_weight_bottom25_threshold": 58.27188724235731
}

Top-q75 stress summary:
                            model_id  n_rows    mae_q75  pinball_mean    iqr_mae
      M05::prior_feature_lgbm_direct    1058 605.877011    219.575063 496.078294
   M05::residual_lgbm_prior_features    1058 607.540468    218.946826 501.218487
    MLP::mlp_residual_prior_features    1058 623.819626    225.692988 492.566477
 M05::residual_lgbm_current_features    1058 634.736172    217.249040 535.855900
  MLP::mlp_residual_current_features    1058 641.957474    225.283272 532.710683
       MLP::mlp_raw_current_features    1058 687.172521    246.488422 546.991990
                M05::lightgbm_direct    1058 690.011190    310.243366 556.894353
      M05::hist_lgbm_ensemble_global    1058 694.091366    253.087222 527.517663
M05::hist_lgbm_ensemble_per_quantile    105

C:\Users\Nick_James\AppData\Local\Temp\ipykernel_24836\2643045463.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = out[col].fillna(False).astype(bool)
C:\Users\Nick_James\AppData\Local\Temp\ipykernel_24836\2643045463.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[col] = out[col].fillna(False).astype(bool)
C:\Users\Nick_James\AppData\Local\Temp\ipykernel_24836\2643045463.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy

## 11. Segment binning utilities

This section creates reusable binning functions for segment-level error analysis. The notebook uses quantile bins for continuous variables and categorical grouping for binary/mode variables.

In [12]:
def safe_qcut(series: pd.Series, q: int, prefix: str) -> pd.Series:
    """Quantile-bin a series while handling repeated values and missing data."""
    x = pd.to_numeric(series, errors="coerce")
    labels = pd.Series("missing", index=series.index, dtype="object")
    valid = x.notna() & np.isfinite(x)
    if valid.sum() == 0:
        return labels

    try:
        binned = pd.qcut(x[valid], q=q, duplicates="drop")
        codes = binned.cat.codes.to_numpy()
        categories = list(binned.cat.categories)
        row_labels = []
        for code in codes:
            if code < 0:
                row_labels.append("missing")
            else:
                row_labels.append(f"{prefix}_{code + 1:02d}: {categories[code]}")
        labels.loc[valid] = row_labels
    except Exception:
        # Fallback to rank-based bins if qcut fails due to too many ties.
        ranks = x[valid].rank(method="average", pct=True)
        edges = np.linspace(0, 1, q + 1)
        codes = np.searchsorted(edges, ranks, side="right") - 1
        codes = np.clip(codes, 0, q - 1)
        labels.loc[valid] = [f"{prefix}_{int(code)+1:02d}" for code in codes]
    return labels


def add_segment_bins(df: pd.DataFrame) -> pd.DataFrame:
    """Add decile and bucket columns used for segment diagnostics."""
    out = df.copy()

    # Compute bins from unique OD-year rows to avoid model repetition influencing quantiles.
    candidate_cols = [
        "true_q75", "true_iqr", "n_fmi_county_pairs", "obs_weight_sum",
        "log1p_total_tons_modes_1_2_5", "log1p_total_tmiles_modes_1_2_5",
    ]
    available_cols = [c for c in candidate_cols if c in out.columns]
    unique_rows = out[KEY_COLUMNS + available_cols].drop_duplicates(KEY_COLUMNS).copy()

    unique_rows["true_q75_decile"] = safe_qcut(unique_rows["true_q75"], 10, "true_q75_decile")
    unique_rows["true_iqr_decile"] = safe_qcut(unique_rows["true_iqr"], 10, "true_iqr_decile")
    unique_rows["n_fmi_county_pairs_quartile"] = safe_qcut(unique_rows["n_fmi_county_pairs"], 4, "n_fmi_quartile")
    unique_rows["obs_weight_sum_quartile"] = safe_qcut(unique_rows["obs_weight_sum"], 4, "obs_weight_quartile")

    if "log1p_total_tons_modes_1_2_5" in unique_rows.columns:
        unique_rows["total_tons_decile"] = safe_qcut(unique_rows["log1p_total_tons_modes_1_2_5"], 10, "tons_decile")
    if "log1p_total_tmiles_modes_1_2_5" in unique_rows.columns:
        unique_rows["total_tmiles_decile"] = safe_qcut(unique_rows["log1p_total_tmiles_modes_1_2_5"], 10, "tmiles_decile")

    bin_cols = [c for c in unique_rows.columns if c.endswith("decile") or c.endswith("quartile")]
    out = out.merge(unique_rows[KEY_COLUMNS + bin_cols], on=KEY_COLUMNS, how="left", validate="many_to_one")

    # Create readable binary/categorical labels.
    for binary_col in ["same_faf_zone", "has_truck_demand", "has_rail_demand", "has_multimodal_demand"]:
        if binary_col in out.columns:
            out[f"{binary_col}_label"] = out[binary_col].fillna(0).astype(int).map({0: f"{binary_col}=0", 1: f"{binary_col}=1"})

    if "n_modes_with_positive_tons" in out.columns:
        out["n_modes_with_positive_tons_label"] = "n_modes=" + out["n_modes_with_positive_tons"].fillna(-1).astype(int).astype(str)

    return out


combined_pred = add_segment_bins(combined_pred)
print("Added segment columns:")
print([c for c in combined_pred.columns if c.endswith("decile") or c.endswith("quartile") or c.endswith("_label")])

Added segment columns:
['true_q75_decile', 'true_iqr_decile', 'n_fmi_county_pairs_quartile', 'obs_weight_sum_quartile', 'total_tons_decile', 'total_tmiles_decile', 'same_faf_zone_label', 'has_truck_demand_label', 'has_rail_demand_label', 'has_multimodal_demand_label', 'n_modes_with_positive_tons_label']


## 12. Model-level and segment-level summaries

This cell computes standard leaderboards and segment summaries. These tables are the main inputs for the final markdown report and for future paper tables.

In [13]:
def segment_summary_for_column(df: pd.DataFrame, segment_col: str, split: str = "test") -> pd.DataFrame:
    """Summarize model performance by a segment column."""
    if segment_col not in df.columns:
        return pd.DataFrame()
    subset = df.loc[df["split"].eq(split) & df[segment_col].notna()].copy()
    if subset.empty:
        return pd.DataFrame()
    summary = summarize_by(subset, [segment_col, "source", "model", "model_id", "variant", "split"])
    return summary.sort_values([segment_col, "pinball_mean", "mae_q75"]).reset_index(drop=True)


segment_columns = [
    "true_q75_decile",
    "true_iqr_decile",
    "n_fmi_county_pairs_quartile",
    "obs_weight_sum_quartile",
    "total_tons_decile",
    "total_tmiles_decile",
    "same_faf_zone_label",
    "has_rail_demand_label",
    "has_multimodal_demand_label",
    "n_modes_with_positive_tons_label",
]

segment_summaries: Dict[str, pd.DataFrame] = {}
for col in segment_columns:
    if col in combined_pred.columns:
        segment_summaries[col] = segment_summary_for_column(combined_pred, col, split="test")
        print(f"{col}: {segment_summaries[col].shape}")

# Add explicit stress and sparse subset summaries.
stress_subset_summary = summarize_by(
    combined_pred.loc[combined_pred["split"].eq("test") & combined_pred["is_test_q75_top10"]],
    ["source", "model", "model_id", "variant", "split"],
).sort_values("mae_q75").reset_index(drop=True)

sparse_subset_summary = summarize_by(
    combined_pred.loc[combined_pred["split"].eq("test") & combined_pred["is_test_sparse_n_fmi_bottom25"]],
    ["source", "model", "model_id", "variant", "split"],
).sort_values("mae_q75").reset_index(drop=True)

print("\nStress top-q75 subset summary:")
print(stress_subset_summary[["model_id", "n_rows", "mae_q75", "pinball_mean", "iqr_mae"]].head(12).to_string(index=False))
print("\nSparse bottom-n_fmi subset summary:")
print(sparse_subset_summary[["model_id", "n_rows", "mae_q75", "pinball_mean", "iqr_mae"]].head(12).to_string(index=False))

true_q75_decile: (140, 24)
true_iqr_decile: (140, 24)
n_fmi_county_pairs_quartile: (56, 24)
obs_weight_sum_quartile: (56, 24)
total_tons_decile: (140, 24)
total_tmiles_decile: (140, 24)
same_faf_zone_label: (28, 24)
has_rail_demand_label: (28, 24)
has_multimodal_demand_label: (28, 24)
n_modes_with_positive_tons_label: (56, 24)

Stress top-q75 subset summary:
                            model_id  n_rows    mae_q75  pinball_mean    iqr_mae
      M05::prior_feature_lgbm_direct    1058 605.877011    219.575063 496.078294
   M05::residual_lgbm_prior_features    1058 607.540468    218.946826 501.218487
    MLP::mlp_residual_prior_features    1058 623.819626    225.692988 492.566477
 M05::residual_lgbm_current_features    1058 634.736172    217.249040 535.855900
  MLP::mlp_residual_current_features    1058 641.957474    225.283272 532.710683
       MLP::mlp_raw_current_features    1058 687.172521    246.488422 546.991990
                M05::lightgbm_direct    1058 690.011190    310.243366 55

## 13. Paired M0.5-vs-MLP row-level comparisons

A paired comparison aligns M0.5 and MLP predictions on the same FAF OD-year rows. This is stronger than comparing aggregate leaderboards because it shows **where** one model improves over the other.

For each pair, the notebook computes row-level deltas such as:

- `delta_pinball_mean_m05_minus_mlp`
- `delta_abs_error_q75_m05_minus_mlp`
- `delta_abs_error_iqr_m05_minus_mlp`

A positive delta means the MLP model has lower error for that row.

In [14]:
def build_pairwise_comparison(
    df: pd.DataFrame,
    pair_name: str,
    m05_model: str,
    mlp_model: str,
    split: str = "test",
) -> pd.DataFrame:
    """Build a row-level paired comparison for one M0.5 model and one MLP model."""
    m05 = df.loc[(df["source"].eq("M05")) & (df["model"].eq(m05_model)) & (df["split"].eq(split))].copy()
    mlp = df.loc[(df["source"].eq("MLP")) & (df["model"].eq(mlp_model)) & (df["split"].eq(split))].copy()

    if m05.empty or mlp.empty:
        warnings.warn(f"Skipping pair {pair_name}: missing M0.5 or MLP rows.")
        return pd.DataFrame()

    metric_cols = [
        "pinball_mean_row", "abs_error_q25", "abs_error_q50", "abs_error_q75", "abs_error_iqr",
        "error_q25", "error_q50", "error_q75", "sample_weight_common",
        "true_q25", "true_q50", "true_q75", "true_iqr", "pred_q25", "pred_q50", "pred_q75", "pred_iqr",
        "n_fmi_county_pairs", "obs_weight_sum", "faf_od",
    ]
    segment_cols = [
        c for c in df.columns
        if c.endswith("decile") or c.endswith("quartile") or c.endswith("_label") or c.startswith("is_test_")
    ]
    keep_cols = KEY_COLUMNS + [c for c in metric_cols + segment_cols if c in df.columns]

    m05_small = m05[keep_cols].add_suffix("_m05")
    mlp_small = mlp[keep_cols].add_suffix("_mlp")

    # Restore key names for merge.
    for key in KEY_COLUMNS:
        m05_small = m05_small.rename(columns={f"{key}_m05": key})
        mlp_small = mlp_small.rename(columns={f"{key}_mlp": key})

    paired = m05_small.merge(mlp_small, on=KEY_COLUMNS, how="inner", validate="one_to_one")
    paired.insert(0, "pair_name", pair_name)
    paired.insert(1, "m05_model", m05_model)
    paired.insert(2, "mlp_model", mlp_model)

    # Positive deltas mean the MLP model has lower error than the M0.5 model.
    for metric in ["pinball_mean_row", "abs_error_q25", "abs_error_q50", "abs_error_q75", "abs_error_iqr"]:
        paired[f"delta_{metric}_m05_minus_mlp"] = paired[f"{metric}_m05"] - paired[f"{metric}_mlp"]
        paired[f"mlp_wins_{metric}"] = paired[f"delta_{metric}_m05_minus_mlp"] > 0

    return paired


def summarize_pairwise_comparison(paired: pd.DataFrame) -> pd.DataFrame:
    """Summarize one or more paired comparison tables."""
    if paired.empty:
        return pd.DataFrame()

    records = []
    for pair_name, group in paired.groupby("pair_name", sort=True):
        w = group.get("sample_weight_common_mlp", pd.Series(1.0, index=group.index))
        row = {
            "pair_name": pair_name,
            "m05_model": group["m05_model"].iloc[0],
            "mlp_model": group["mlp_model"].iloc[0],
            "n_rows": int(len(group)),
        }
        for metric in ["pinball_mean_row", "abs_error_q25", "abs_error_q50", "abs_error_q75", "abs_error_iqr"]:
            delta_col = f"delta_{metric}_m05_minus_mlp"
            win_col = f"mlp_wins_{metric}"
            row[f"mean_delta_{metric}"] = weighted_mean(group[delta_col])
            row[f"weighted_mean_delta_{metric}"] = weighted_mean(group[delta_col], w)
            row[f"mlp_win_rate_{metric}"] = weighted_mean(group[win_col].astype(float))
            row[f"weighted_mlp_win_rate_{metric}"] = weighted_mean(group[win_col].astype(float), w)
        records.append(row)
    return pd.DataFrame(records)


pairwise_tables = []
for spec in PAIR_DEFINITIONS:
    pair_df = build_pairwise_comparison(
        combined_pred,
        pair_name=spec["pair_name"],
        m05_model=spec["m05_model"],
        mlp_model=spec["mlp_model"],
        split="test",
    )
    if not pair_df.empty:
        pairwise_tables.append(pair_df)

paired_test = pd.concat(pairwise_tables, ignore_index=True) if pairwise_tables else pd.DataFrame()
paired_summary_test = summarize_pairwise_comparison(paired_test)

print("Paired comparison summary:")
if paired_summary_test.empty:
    print("No valid pairwise comparisons were built.")
else:
    display_cols = [
        "pair_name", "n_rows", "mean_delta_pinball_mean_row", "mlp_win_rate_pinball_mean_row",
        "mean_delta_abs_error_q75", "mlp_win_rate_abs_error_q75", "mean_delta_abs_error_iqr",
    ]
    print(paired_summary_test[display_cols].to_string(index=False))

Paired comparison summary:
                                     pair_name  n_rows  mean_delta_pinball_mean_row  mlp_win_rate_pinball_mean_row  mean_delta_abs_error_q75  mlp_win_rate_abs_error_q75  mean_delta_abs_error_iqr
      Historical_prior_vs_MLP_residual_current   10574                    17.066543                       0.687441                 17.107436                    0.566389                  9.072253
        Historical_prior_vs_MLP_residual_prior   10574                    14.094467                       0.731511                 13.672837                    0.587573                 10.470442
M05_prior_feature_direct_vs_MLP_residual_prior   10574                    -7.600970                       0.381029                 -9.944193                    0.447607                  6.806365
    M05_residual_prior_vs_MLP_residual_current   10574                    -4.745782                       0.419141                 -3.941307                    0.479951                  8.24988

## 14. Segment-level pairwise comparisons

This cell summarizes paired M0.5-vs-MLP deltas by important segments, such as true q75 deciles and sparse observation-support buckets.

These tables answer questions like:

- Does MLP beat M0.5 more often in high-q75 stress rows?
- Does M0.5 remain stronger in sparse-label rows?
- Is the MLP advantage concentrated in certain demand regimes?

In [15]:
def segment_pairwise_summary(paired: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    """Summarize paired deltas by pair name and a row-level segment."""
    if paired.empty:
        return pd.DataFrame()

    m05_segment_col = f"{segment_col}_m05"
    mlp_segment_col = f"{segment_col}_mlp"
    if m05_segment_col in paired.columns:
        seg_col = m05_segment_col
    elif mlp_segment_col in paired.columns:
        seg_col = mlp_segment_col
    else:
        return pd.DataFrame()

    records = []
    for (pair_name, segment_value), group in paired.groupby(["pair_name", seg_col], dropna=False, sort=True):
        w = group.get("sample_weight_common_mlp", pd.Series(1.0, index=group.index))
        row = {
            "pair_name": pair_name,
            "segment_column": segment_col,
            "segment_value": segment_value,
            "n_rows": int(len(group)),
            "mean_delta_pinball": weighted_mean(group["delta_pinball_mean_row_m05_minus_mlp"]),
            "weighted_mean_delta_pinball": weighted_mean(group["delta_pinball_mean_row_m05_minus_mlp"], w),
            "mlp_win_rate_pinball": weighted_mean(group["mlp_wins_pinball_mean_row"].astype(float)),
            "mean_delta_q75_abs_error": weighted_mean(group["delta_abs_error_q75_m05_minus_mlp"]),
            "mlp_win_rate_q75_abs_error": weighted_mean(group["mlp_wins_abs_error_q75"].astype(float)),
            "mean_delta_iqr_abs_error": weighted_mean(group["delta_abs_error_iqr_m05_minus_mlp"]),
            "mlp_win_rate_iqr_abs_error": weighted_mean(group["mlp_wins_abs_error_iqr"].astype(float)),
        }
        records.append(row)
    return pd.DataFrame(records)


pair_segment_columns = [
    "true_q75_decile",
    "true_iqr_decile",
    "n_fmi_county_pairs_quartile",
    "obs_weight_sum_quartile",
    "total_tons_decile",
    "same_faf_zone_label",
    "has_rail_demand_label",
    "has_multimodal_demand_label",
    "n_modes_with_positive_tons_label",
    "is_test_q75_top10",
    "is_test_sparse_n_fmi_bottom25",
]

pair_segment_summaries: Dict[str, pd.DataFrame] = {}
for col in pair_segment_columns:
    summary = segment_pairwise_summary(paired_test, col)
    if not summary.empty:
        pair_segment_summaries[col] = summary
        print(f"{col}: {summary.shape}")

true_q75_decile: (40, 11)
true_iqr_decile: (40, 11)
n_fmi_county_pairs_quartile: (16, 11)
obs_weight_sum_quartile: (16, 11)
total_tons_decile: (40, 11)
same_faf_zone_label: (8, 11)
has_rail_demand_label: (8, 11)
has_multimodal_demand_label: (8, 11)
n_modes_with_positive_tons_label: (16, 11)
is_test_q75_top10: (8, 11)
is_test_sparse_n_fmi_bottom25: (8, 11)


## 15. Top disagreement and failure cases

This section saves row-level cases where:

- MLP improves most over M0.5
- M0.5 improves most over MLP
- both models have large q75 errors

These files are useful for manual inspection and for identifying future graph/topology/disruption feature needs.

In [16]:
def top_cases_for_pair(
    paired: pd.DataFrame,
    pair_name: str,
    metric_delta_col: str,
    top_n: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return top MLP-win and top M0.5-win cases for one pair and one delta metric."""
    subset = paired.loc[paired["pair_name"].eq(pair_name)].copy()
    if subset.empty:
        return pd.DataFrame(), pd.DataFrame()

    mlp_wins = subset.sort_values(metric_delta_col, ascending=False).head(top_n).copy()
    m05_wins = subset.sort_values(metric_delta_col, ascending=True).head(top_n).copy()
    return mlp_wins, m05_wins


def both_fail_cases(paired: pd.DataFrame, pair_name: str, top_n: int) -> pd.DataFrame:
    """Return cases where both M0.5 and MLP have large q75 errors."""
    subset = paired.loc[paired["pair_name"].eq(pair_name)].copy()
    if subset.empty:
        return pd.DataFrame()

    subset["both_model_q75_abs_error_mean"] = (subset["abs_error_q75_m05"] + subset["abs_error_q75_mlp"]) / 2.0
    return subset.sort_values("both_model_q75_abs_error_mean", ascending=False).head(top_n).copy()


top_case_tables: Dict[str, pd.DataFrame] = {}
for pair_name in paired_test["pair_name"].drop_duplicates().tolist() if not paired_test.empty else []:
    mlp_wins, m05_wins = top_cases_for_pair(
        paired_test,
        pair_name=pair_name,
        metric_delta_col="delta_abs_error_q75_m05_minus_mlp",
        top_n=CFG.top_case_count,
    )
    both_fail = both_fail_cases(paired_test, pair_name=pair_name, top_n=CFG.top_case_count)
    top_case_tables[f"{pair_name}__mlp_wins_q75"] = mlp_wins
    top_case_tables[f"{pair_name}__m05_wins_q75"] = m05_wins
    top_case_tables[f"{pair_name}__both_fail_q75"] = both_fail

print("Generated top-case tables:")
for name, table in top_case_tables.items():
    print(f"  {name}: {table.shape}")

Generated top-case tables:
  M05_residual_prior_vs_MLP_residual_current__mlp_wins_q75: (100, 87)
  M05_residual_prior_vs_MLP_residual_current__m05_wins_q75: (100, 87)
  M05_residual_prior_vs_MLP_residual_current__both_fail_q75: (100, 88)
  M05_prior_feature_direct_vs_MLP_residual_prior__mlp_wins_q75: (100, 87)
  M05_prior_feature_direct_vs_MLP_residual_prior__m05_wins_q75: (100, 87)
  M05_prior_feature_direct_vs_MLP_residual_prior__both_fail_q75: (100, 88)
  Historical_prior_vs_MLP_residual_current__mlp_wins_q75: (100, 87)
  Historical_prior_vs_MLP_residual_current__m05_wins_q75: (100, 87)
  Historical_prior_vs_MLP_residual_current__both_fail_q75: (100, 88)
  Historical_prior_vs_MLP_residual_prior__mlp_wins_q75: (100, 87)
  Historical_prior_vs_MLP_residual_prior__m05_wins_q75: (100, 87)
  Historical_prior_vs_MLP_residual_prior__both_fail_q75: (100, 88)


## 16. Origin and destination residual-bias diagnostics

This section identifies FAF origins and destinations where each model systematically overpredicts or underpredicts q75. These diagnostics are useful for deciding where graph structure, topology features, or disruption features may be most valuable.

In [17]:
def zone_bias_summary(df: pd.DataFrame, zone_col: str, split: str = "test", top_n: int = 25) -> pd.DataFrame:
    """Summarize q75 bias and absolute q75 error by origin or destination FAF zone."""
    if zone_col not in df.columns:
        return pd.DataFrame()
    subset = df.loc[df["split"].eq(split)].copy()
    records = []
    for (model_id, zone), group in subset.groupby(["model_id", zone_col], dropna=False, sort=True):
        records.append({
            "model_id": model_id,
            "zone_column": zone_col,
            "zone": zone,
            "n_rows": int(len(group)),
            "mean_q75_bias_pred_minus_true": weighted_mean(group["error_q75"]),
            "mean_abs_error_q75": weighted_mean(group["abs_error_q75"]),
            "mean_pinball": weighted_mean(group["pinball_mean_row"]),
            "mean_true_q75": weighted_mean(group["true_q75"]),
        })
    summary = pd.DataFrame(records)
    if summary.empty:
        return summary

    # Keep zones with enough rows to avoid overinterpreting tiny groups.
    min_rows = max(5, int(summary["n_rows"].quantile(0.10)))
    summary = summary.loc[summary["n_rows"] >= min_rows].copy()
    return summary.sort_values(["model_id", "mean_abs_error_q75"], ascending=[True, False]).reset_index(drop=True)


origin_bias_summary = zone_bias_summary(combined_pred, "faf_orig", split="test")
destination_bias_summary = zone_bias_summary(combined_pred, "faf_dest", split="test")

print("Origin bias summary shape:", origin_bias_summary.shape)
print("Destination bias summary shape:", destination_bias_summary.shape)

Origin bias summary shape: (1344, 8)
Destination bias summary shape: (1316, 8)


## 17. Optional diagnostic plots

The notebook produces simple matplotlib plots when matplotlib is available. Plots are helpful for presentations, but the CSV/Parquet tables are the primary analysis artifacts.

In [18]:
def save_bar_plot(df: pd.DataFrame, label_col: str, value_col: str, title: str, output_path: Path, top_n: int = 12) -> None:
    """Save a horizontal bar plot using matplotlib defaults."""
    if not HAS_MATPLOTLIB or not CFG.make_plots:
        return
    plot_df = df.sort_values(value_col, ascending=True).head(top_n).copy()
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(plot_df))))
    ax.barh(plot_df[label_col].astype(str), plot_df[value_col].astype(float))
    ax.set_title(title)
    ax.set_xlabel(value_col)
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


def save_segment_line_plot(summary: pd.DataFrame, segment_col: str, value_col: str, title: str, output_path: Path, focus_models: Sequence[str]) -> None:
    """Save a model-by-segment line plot using matplotlib defaults."""
    if not HAS_MATPLOTLIB or not CFG.make_plots or summary.empty:
        return
    if segment_col not in summary.columns:
        return

    plot_df = summary.loc[summary["model_id"].isin(focus_models)].copy()
    if plot_df.empty:
        return

    fig, ax = plt.subplots(figsize=(12, 6))
    for model_id, group in plot_df.groupby("model_id", sort=True):
        group = group.sort_values(segment_col)
        ax.plot(group[segment_col].astype(str), group[value_col].astype(float), marker="o", label=model_id)
    ax.set_title(title)
    ax.set_xlabel(segment_col)
    ax.set_ylabel(value_col)
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


if HAS_MATPLOTLIB and CFG.make_plots:
    save_bar_plot(
        leaderboard_test,
        label_col="model_id",
        value_col="pinball_mean",
        title="Test Pinball Mean Leaderboard",
        output_path=PATHS.plot_dir / "leaderboard_test_pinball_mean.png",
    )
    save_bar_plot(
        leaderboard_test,
        label_col="model_id",
        value_col="mae_q75",
        title="Test q75 MAE Leaderboard",
        output_path=PATHS.plot_dir / "leaderboard_test_mae_q75.png",
    )
    if "true_q75_decile" in segment_summaries:
        save_segment_line_plot(
            segment_summaries["true_q75_decile"],
            segment_col="true_q75_decile",
            value_col="mae_q75",
            title="Test q75 MAE by True q75 Decile",
            output_path=PATHS.plot_dir / "mae_q75_by_true_q75_decile.png",
            focus_models=DEFAULT_FOCUS_MODELS,
        )
    if "true_iqr_decile" in segment_summaries:
        save_segment_line_plot(
            segment_summaries["true_iqr_decile"],
            segment_col="true_iqr_decile",
            value_col="iqr_mae",
            title="Test IQR MAE by True IQR Decile",
            output_path=PATHS.plot_dir / "iqr_mae_by_true_iqr_decile.png",
            focus_models=DEFAULT_FOCUS_MODELS,
        )
    print("Plots saved to:", PATHS.plot_dir)
else:
    print("Plot generation skipped.")

Plots saved to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_vs_mlp_error_analysis_v1_notebook\east_plus_gulf\plots


## 18. Save all analysis artifacts

This cell writes all outputs to the dedicated analysis directory.

Primary outputs:

- combined model leaderboard
- full model-by-split metrics
- segment-level summaries
- paired row-level M0.5-vs-MLP comparisons
- top disagreement cases
- origin/destination bias diagnostics
- an artifact manifest

In [19]:
def write_dataframe(df: pd.DataFrame, path: Path) -> None:
    """Write a dataframe to CSV, creating parent directories if needed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def safe_filename(name: str) -> str:
    """Create a filesystem-safe filename fragment."""
    return "".join(ch if ch.isalnum() or ch in {"_", "-"} else "_" for ch in name)


# Core model-level outputs.
write_dataframe(metrics_by_model_split, PATHS.table_dir / "metrics_by_model_split.csv")
write_dataframe(leaderboard_test, PATHS.table_dir / "leaderboard_test_combined.csv")
write_dataframe(stress_subset_summary, PATHS.table_dir / "stress_top_q75_subset_summary.csv")
write_dataframe(sparse_subset_summary, PATHS.table_dir / "sparse_bottom_n_fmi_subset_summary.csv")

# Segment summaries.
for segment_col, table in segment_summaries.items():
    if not table.empty:
        write_dataframe(table, PATHS.table_dir / f"segment_summary__{safe_filename(segment_col)}.csv")

# Pairwise comparison outputs.
if not paired_summary_test.empty:
    write_dataframe(paired_summary_test, PATHS.table_dir / "paired_summary_test.csv")
if not paired_test.empty:
    paired_test_path = PATHS.table_dir / "paired_row_level_test.parquet"
    paired_test.to_parquet(paired_test_path, index=False)

for segment_col, table in pair_segment_summaries.items():
    if not table.empty:
        write_dataframe(table, PATHS.table_dir / f"paired_segment_summary__{safe_filename(segment_col)}.csv")

# Top case outputs.
for name, table in top_case_tables.items():
    if not table.empty:
        write_dataframe(table, PATHS.table_dir / f"top_cases__{safe_filename(name)}.csv")

# Zone-bias diagnostics.
write_dataframe(origin_bias_summary, PATHS.table_dir / "origin_faf_q75_bias_summary.csv")
write_dataframe(destination_bias_summary, PATHS.table_dir / "destination_faf_q75_bias_summary.csv")

# Save normalized prediction diagnostics for reproducibility.
combined_pred.to_parquet(PATHS.analysis_output_dir / "combined_normalized_predictions_val_test.parquet", index=False)

artifact_manifest = {
    "notebook": "M0.5_vs_MLP_ErrorAnalysis.ipynb",
    "config": {
        **asdict(CFG),
        "data_root": str(CFG.data_root),
    },
    "paths": {
        "supervised_path": str(PATHS.supervised_path),
        "manifest_path": str(PATHS.manifest_path),
        "m05_prediction_path": str(m05_prediction_path) if m05_prediction_path is not None else None,
        "mlp_prediction_path": str(mlp_prediction_path) if mlp_prediction_path is not None else None,
        "analysis_output_dir": str(PATHS.analysis_output_dir),
    },
    "diagnostic_thresholds": diagnostic_thresholds,
    "combined_shape": list(combined_pred.shape),
    "model_ids": sorted(combined_pred["model_id"].dropna().unique().tolist()),
    "table_outputs": sorted(str(p) for p in PATHS.table_dir.glob("*.csv")),
    "plot_outputs": sorted(str(p) for p in PATHS.plot_dir.glob("*.png")),
}

manifest_path = PATHS.analysis_output_dir / "analysis_artifact_manifest.json"
with manifest_path.open("w", encoding="utf-8") as fp:
    json.dump(artifact_manifest, fp, indent=2, ensure_ascii=False)

print("Saved analysis outputs to:", PATHS.analysis_output_dir)
print("Saved artifact manifest:", manifest_path)

Saved analysis outputs to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_vs_mlp_error_analysis_v1_notebook\east_plus_gulf
Saved artifact manifest: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_vs_mlp_error_analysis_v1_notebook\east_plus_gulf\analysis_artifact_manifest.json


## 19. Generate a compact markdown report

This final report summarizes the strongest models, key paired comparisons, and diagnostic thresholds. It is intended as a quick-readable experiment log, not as a substitute for the detailed CSV tables.

In [20]:
def dataframe_to_markdown_table(df: pd.DataFrame, columns: Sequence[str], max_rows: int = 10) -> str:
    """Convert selected dataframe columns to a compact markdown table."""
    if df.empty:
        return "No rows available."
    view = df.loc[:, [c for c in columns if c in df.columns]].head(max_rows).copy()
    return view.to_markdown(index=False)


def write_markdown_report() -> Path:
    """Write a compact markdown summary report for this error analysis run."""
    report_path = PATHS.report_dir / "m05_vs_mlp_error_analysis_report.md"

    leaderboard_cols = [
        "rank_by_test_pinball", "model_id", "pinball_mean", "weighted_pinball_mean",
        "mae_q50", "mae_q75", "iqr_mae", "raw_crossing_rate",
    ]
    paired_cols = [
        "pair_name", "n_rows", "mean_delta_pinball_mean_row", "mlp_win_rate_pinball_mean_row",
        "mean_delta_abs_error_q75", "mlp_win_rate_abs_error_q75", "mean_delta_abs_error_iqr",
    ]

    lines = []
    lines.append("# M0.5 vs MLP Error Analysis Report")
    lines.append("")
    lines.append("## Input artifacts")
    lines.append(f"- M0.5 prediction file: `{m05_prediction_path}`")
    lines.append(f"- MLP prediction file: `{mlp_prediction_path}`")
    lines.append(f"- Supervised table: `{PATHS.supervised_path}`")
    lines.append("")
    lines.append("## Diagnostic thresholds")
    for key, value in diagnostic_thresholds.items():
        lines.append(f"- `{key}`: {value:.6f}")
    lines.append("")
    lines.append("## Test leaderboard")
    lines.append(dataframe_to_markdown_table(leaderboard_test, leaderboard_cols, max_rows=15))
    lines.append("")
    lines.append("## Paired M0.5-vs-MLP comparisons")
    lines.append(dataframe_to_markdown_table(paired_summary_test, paired_cols, max_rows=20))
    lines.append("")
    lines.append("## Stress subset summary")
    lines.append(dataframe_to_markdown_table(stress_subset_summary, ["model_id", "n_rows", "mae_q75", "pinball_mean", "iqr_mae"], max_rows=15))
    lines.append("")
    lines.append("## Sparse-label subset summary")
    lines.append(dataframe_to_markdown_table(sparse_subset_summary, ["model_id", "n_rows", "mae_q75", "pinball_mean", "iqr_mae"], max_rows=15))
    lines.append("")
    lines.append("## Interpretation checklist")
    lines.append("- Identify whether M0.5 remains stronger on overall pinball.")
    lines.append("- Identify whether MLP residual models improve IQR, stress-q75, or sparse-label subsets.")
    lines.append("- Inspect top disagreement cases to decide whether topology or disruption features are needed.")
    lines.append("- Use cold-OD and cold-destination splits before making final graph-model claims.")

    report_path.write_text("\n".join(lines), encoding="utf-8")
    return report_path


report_path = write_markdown_report()
print("Markdown report saved to:", report_path)

ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.